<a href="https://colab.research.google.com/github/Baptistecaille/JEPA-PC/blob/main/no-ema-sigreg-OHc9i/main.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# PC-JEPA — Predictive Coding + JEPA on Moving MNIST

**Before running:** `Runtime → Change runtime type → TPU v2` (free tier) or TPU v3/v4.  
JAX with TPU support is pre-installed on Colab TPU runtimes — no extra pip install required.

Architecture : CNN encoder → PC hierarchy (Loop 1, inference) → Transformer predictor → JEPA loss (Loop 2, learning).

> **TPU note** — JAX holds an exclusive lock on the TPU for the kernel process.
> JAX-heavy cells use `%run` (IPython magic) so they execute **in the current
> kernel process** and share the existing lock. `!python` would fork a new
> process that cannot access the locked TPU.

Cells in order:
1. Verify TPU backend (pre-installed JAX)
2. Clone repo
3. Install optax (only missing dependency)
4. Download MNIST
5. Sanity checks
6. `check_tconv` — convergence diagnostic
7. Exp2 — sample efficiency curve (PC-JEPA vs Transformer)

In [2]:
# ── 1. Verify pre-installed JAX sees the TPU ──────────────────────────────────
# Colab TPU runtime ships JAX with TPU support already installed.
# No pip install needed — just confirm the backend before proceeding.
import jax

print(f"JAX version : {jax.__version__}")
print(f"Backend     : {jax.default_backend()}")
print(f"Devices     : {jax.devices()}")

JAX version : 0.7.2
Backend     : cpu
Devices     : [CpuDevice(id=0)]


In [3]:
# ── 3. Clone repo ─────────────────────────────────────────────────────────────
import os

REPO_URL    = "https://github.com/Baptistecaille/JEPA-PC.git"
BRANCH      = "claude/no-ema-sigreg-OHc9i"   # change to 'main' for stable release
REPO_DIR    = "JEPA-PC"

if not os.path.isdir(REPO_DIR):
    !git clone --branch {BRANCH} {REPO_URL} {REPO_DIR}
else:
    print(f"{REPO_DIR} already exists — pulling latest changes")
    !git -C {REPO_DIR} pull origin {BRANCH}

%cd {REPO_DIR}
!git log --oneline -3

Cloning into 'JEPA-PC'...
remote: Enumerating objects: 248, done.
remote: Counting objects: 100% (248/248), done.
remote: Compressing objects: 100% (181/181), done.
remote: Total 248 (delta 150), reused 141 (delta 64), pack-reused 0 (from 0)
Receiving objects: 100% (248/248), 126.18 KiB | 2.43 MiB/s, done.
Resolving deltas: 100% (150/150), done.
/content/JEPA-PC
6312332 (HEAD -> claude/no-ema-sigreg-OHc9i, origin/claude/no-ema-sigreg-OHc9i) Remove jax[tpu] pip install — use Colab's pre-installed JAX
b035a47 Fix TPU lock contention: !python → %run in JAX cells
344dd17 Rewrite main.ipynb for Colab TPU


In [5]:
# ── 3. Install project dependencies ───────────────────────────────────────────
# Colab TPU runtime pre-installs: jax, jaxlib (with TPU), numpy, matplotlib,
# tensorflow (used to download MNIST). Only optax needs installing.
!pip install -q optax tensorflow

In [7]:
# ── 5. Pre-download MNIST ─────────────────────────────────────────────────────
# The data loader reads from ~/.keras/datasets/mnist.npz.
# tensorflow.keras is pre-installed on Colab and handles the download.
import tensorflow as tf

(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()
print(f"MNIST loaded — train: {x_train.shape}, test: {x_test.shape}")

MNIST loaded — train: (60000, 28, 28), test: (10000, 28, 28)


In [8]:
# ── 6. Sanity checks — all modules ────────────────────────────────────────────
# %run executes in the current kernel process (shares the TPU lock).
# !python would spawn a new process → "TPU already in use" error.
%run run_all.py --mode sanity


SANITY CHECKS — vérification de tous les modules

[Sanity] data.moving_mnist — début
  [D1] ✓ context: (4, 10, 64, 64, 1), target: (4, 5, 64, 64, 1)
  [D2] ✓ valeurs dans [0, 1]
  [D3] ✓ context et target temporellement distincts
  [D4] ✓ reproductibilité avec même seed
[Sanity] data.moving_mnist — OK

[Sanity] models.encoder — début
  [E1] ✓ shape: (2, 4, 256)
  [E2] ✓ pas de NaN
  [E3] ✓ JIT-compilable
  [E4] ✓ gradients calculables, pas de NaN
  [E5] ✓ poids différents selon le seed
[Sanity] models.encoder — OK

[Sanity] models.pc_nodes — début
  [I1] ✓ free_energy différentiable par rapport à hierarchy_state
  [I2] erreur avant: 0.3195, après 1 pas: 0.3194
  [I3] ✓ JIT-compilable, T_conv=500, err=0.284830
  [I4] ✓ poids inchangés pendant l'inférence
  [I5] ✓ T_conv=500 ≤ pc_max_iter=500
[Sanity] models.pc_nodes — OK

[Sanity] models.predictor (Transformer) — début
  [P1] ✓ shape: (2, 5, 256)
  [P2] ✓ pas de NaN
  [P3] ✓ JIT-compilable
  [P4] ✓ gradients calculables
  [P5] Paramètr

In [ ]:
# ── 7. T_conv diagnostic ──────────────────────────────────────────────────────
# Expected: T_conv < 500, MSE final < pc_tol=0.1, 100% des batches convergent.
%run check_tconv.py

CHECK T_CONV — après fix compute_max_error (L∞ → MSE)
  pc_tol=0.1  pc_max_iter=500

── n=100 ──────────────────────────────────────────
  step=  0  loss=10.4039  T_conv(train)=500  pc_err=0.328203
  step= 50  loss=1.1140  T_conv(train)=500  pc_err=0.326753


In [ ]:
# ── 8. Exp2 — sample efficiency (PC-JEPA vs Transformer) ──────────────────────
# n ∈ (100, 500, 1000, 2000, 5000, 10000) × 2 seeds × 2 models = 24 runs.
# On TPU v2 (8 cores): ~30–60 min depending on n=10000.
%run run_all.py --mode exp2

In [ ]:
# ── 9. Plot results ────────────────────────────────────────────────────────────
import json, glob, os
import numpy as np
import matplotlib.pyplot as plt

# Load most recent exp2 result file
files = sorted(glob.glob("results/exp2_*.json"))
assert files, "No exp2 results found — run cell 8 first."
with open(files[-1]) as f:
    results = json.load(f)
print(f"Loaded: {files[-1]}")

NS     = (100, 500, 1000, 2000, 5000, 10000)
SEEDS  = (42, 137)

pc_means, pc_stds = [], []
tr_means, tr_stds = [], []

for n in NS:
    pc_vals = [results.get(f"pc_jepa_n{n}_seed{s}", {}).get("nmse", float("nan")) for s in SEEDS]
    tr_vals = [results.get(f"transformer_n{n}_seed{s}", {}).get("nmse", float("nan")) for s in SEEDS]
    pc_means.append(np.nanmean(pc_vals))
    pc_stds.append(np.nanstd(pc_vals))
    tr_means.append(np.nanmean(tr_vals))
    tr_stds.append(np.nanstd(tr_vals))

fig, ax = plt.subplots(figsize=(7, 4))
ax.errorbar(NS, pc_means, yerr=pc_stds, marker="o", label="PC-JEPA (ours)", capsize=3)
ax.errorbar(NS, tr_means, yerr=tr_stds, marker="s", linestyle="--",
            label="Transformer baseline", capsize=3)
ax.set_xscale("log")
ax.set_xlabel("Training samples n")
ax.set_ylabel("NMSE (↓)")
ax.set_title("Sample efficiency — PC-JEPA vs Transformer (Moving MNIST)")
ax.legend()
ax.grid(True, which="both", alpha=0.3)
plt.tight_layout()
plt.savefig("results/sample_efficiency.png", dpi=150)
plt.show()
print("Saved: results/sample_efficiency.png")